# Todo
- [ ] SST muss > 15°C sein vor initialem Schlüpfen

In [1]:
import datetime as dt
# from datetime import datetime, timedelta, date
# import dask
import numpy as np
from numpy import datetime64 as dt
from numpy import timedelta64 as td
from pathlib import Path


import xarray as xr
# from time import time
import warnings

import pandas as pd

warnings.simplefilter("ignore")

# Set Parameters

In [2]:
# Parameters
year = 2018
site = 1

lon_min = 0
lon_max = 30
lat_min = 52
lat_max = 62
bin_size = 0.25

isPapermill = False

# Functions

In [3]:
# Paths
trajectory_path = f"/gxfs_work/geomar/smomw597/2025_copepods/output/Trajectories/{year}/"
out_path = f"/gxfs_work/geomar/smomw597/2025_copepods/output/dispersal/{year}/"

In [4]:
file = Path(
    trajectory_path +
    f"PPmill_Nested_{year}0601-{year}1130_dt15min_site{site:02d}_d0m-25m_N1000_seed123.zarr"
)
ds_trajectories = xr.open_zarr(file)


In [5]:
n_trajectories = ds_trajectories.trajectory.shape[0]
trajectories = np.arange(0, n_trajectories, 1000)
print(n_trajectories/1000)
export_vars = ["lat", "lon", "time", "T"]

lonbins = np.arange(lon_min, lon_max+bin_size, bin_size)
latbins = np.arange(lat_min, lat_max+bin_size, bin_size)
print("lon:", lonbins[0],"-", lonbins[-1],", n=",lonbins.shape[0])
print("lat:", latbins[0],"-", latbins[-1],", n=",latbins.shape[0])
lonbins_centers = (lonbins[1:]+lonbins[:-1])/2
latbins_centers = (latbins[1:]+latbins[:-1])/2
dim_dict = {"lon":lonbins_centers, "lat":latbins_centers}

182.0
lon: 0.0 - 30.0 , n= 121
lat: 52.0 - 62.0 , n= 41


In [6]:
age_release_date_array = []
data_arrays = []
release_dates_array = []
for traj in trajectories:
    # Get the trajactories that were released each day
    if isPapermill == True:
        trajectory_slice = slice(traj,traj+1000)
    else:
        trajectory_slice = traj
    ds_bins = ds_trajectories.isel(trajectory=trajectory_slice).get(export_vars)
    nanmask = ds_bins.time.notnull().compute() # mask for all NaT
    # Group the trajactories into groups by the date
    ds_bins = ds_bins.where(nanmask, drop=True).groupby("time.date")#.resample(time="D")
    # Get the keys to reference the single dataarrays
    ds_keys = list(ds_bins.groups.keys())
    # read the date of the first group(date), to get the release_date
    release_date = dt(ds_keys[0])
    # Check, if the release day is within the time frame, else break
    if release_date >= dt(f"{year}-11-01"):
        break
    try:
        hatch_temp = ds_bins[ds_keys[0]].T.max().values
        if hatch_temp > 12:
            # Check if the group has the same lifespan as the others, 
            # if not elongate it (only necessary for testing)
            if len(ds_keys) != 29:
                n = 29-len(ds_keys)
                for i in range(n):
                    ds_keys.append(ds_keys[-1]+1)
            age_release_date_array = []
            # Loop throug each day in the life of the trajectories by utilizing the keys from above
            for date in ds_keys:
                try:
                    # Get all the positional data for the day and make a 2d hist out of it...
                    hist, xedges, yedges = np.histogram2d(
                        ds_bins[date].lon.values,
                        ds_bins[date].lat.values,
                        bins=(lonbins, latbins),
                    )
                except:
                    # if date is empty (so no data) fill with NaN array
                    # (only necessary for testing)
                    hist = np.full_like(hist, np.nan) # hist = np.zeros_like(hist)
                    print(f"for release day {release_date} at {date}")

                # Put the data into a data array, with the lat lon as dimensions
                da = xr.DataArray(
                    hist,
                    dims=dim_dict, coords=dim_dict,
                    name=f"hist-{year}-{site:02d}",
                ).expand_dims({ # Expand the dimensions by site, age and release_date
                    "site":[site], "date": [dt(date)], "release_date": [release_date],
                })
                # put the data array into a list, so it can be concatenated later
                age_release_date_array.append(da)
            try:
                # Concatenate the data arrays of this release day
                # and write it into a list and add age as a coordinate
                single_release_date_data = xr.concat(
                    age_release_date_array, dim="date"
                    ).assign_coords({"age": ("date", range(29))})
                release_dates_array.append(single_release_date_data)
            except:
                print(f"There is no data for release day {release_date}")
                break
    except:
        print("Something went completely wrong.",release_date, date)
# Concatenate the list containing all the data arrays of all release_dates
daily_data = xr.concat(release_dates_array, dim="release_date")
# print the size of the complete data array
print("Size of the data array is",daily_data.nbytes/1e9,"GB")

Size of the data array is 1.0127232 GB


In [10]:
# save the data array to a file
# if isPapermill == True:
daily_data.to_netcdf(
    Path(out_path+f"ds_dispersal_{year}_s{site:02d}.nc")
)

In [8]:
# ### LABOR ###

# age_release_date_array = []
# data_arrays = []
# release_dates_array = []
# for traj in trajectories[:20]:
#     # Get the trajactories that were released each day
#     trajectory_slice = traj
#     ds_bins = ds_trajectories.isel(trajectory=trajectory_slice).get(export_vars)
#     nanmask = ds_bins.time.notnull().compute() # mask for all NaT
#     # Group the trajactories into groups by the date
#     ds_bins = ds_bins.where(nanmask, drop=True).resample(time="D")
#     # Get the keys to reference the single dataarrays
#     ds_keys = list(ds_bins.groups.keys())
#     # read the date of the first group(date), to get the release_date
#     release_date = ds_keys[0]
#     # Check, if the release day is within the time frame, else break
#     if release_date >= dt(f"{year}-11-01"):
#         break
#     hatch_temp = ds_bins[release_date].T.max().values
#     if hatch_temp > 12:
#         age_release_date_array = []
#         # Loop throug each day in the life of the trajectories by utilizing the keys from above
#         for date in ds_keys:
#             try:
#                 # Get all the positional data for the day and make a 2d hist out of it...
#                 hist, xedges, yedges = np.histogram2d(
#                     ds_bins[date].lon.values,
#                     ds_bins[date].lat.values,
#                     bins=(lonbins, latbins),
#                 )
#             except:
#                 # if date is empty (so no data) fill with NaN array (only necessary for testing)
#                 hist = np.full_like(hist, np.nan) # hist = np.zeros_like(hist)
#                 print(f"for release day {release_date} at {date}")
#             # Put the data into a data array, with the lat lon as dimensions
#             da = xr.DataArray(
#                 hist,
#                 dims=dim_dict, coords=dim_dict,
#                 name=f"hist-{year}-{site:02d}",
#             ).expand_dims({ # Expand the dimensions by site, age and release_date
#                 "site":[site], "date": [date], "release_date": [release_date],
#             })
#             # put the data array into a list, so it can be concatenated later
#             age_release_date_array.append(da)
#         try:
#             # Concatenate the data arrays of this release day and write it into a list
#             single_release_date_data = xr.concat(
#                 age_release_date_array, dim="date").assign_coords({"age": ("date", range(29))})
#             release_dates_array.append(single_release_date_data)
#         except:
#             print(f"There is no data for release day {release_date}")
#             break
# # Concatenate the list containing all the data arrays of all release_dates
# daily_data = xr.concat(release_dates_array, dim="release_date")
# # print the size of the complete data array
# print("Size of the data array is",daily_data.nbytes/1e9,"GB")